# Sentiment & Clarity EDA Outputs

This notebook is optimized for a faster presentation workflow using `data/cleaned_yearly_announcements2.csv` instead of the full `data/FINAL.csv` universe.

What changed:
- Uses the smaller yearly-buyback dataset directly.
- Skips the expensive full-universe transcript scan.
- Uses a fast default configuration for full-transcript sentiment.
- Makes real component-based Q&A relevance optional so the default run stays practical on a laptop.

Main outputs:
- Sentiment on buyback-specific language with FinBERT.
- Clarity via specificity, hedge density, and modified FOG.
- Optional Q&A relevance if you explicitly enable it.
- Presentation-ready PNG/CSV outputs under `outputs/eda/`.


In [1]:
from __future__ import annotations

from collections import defaultdict
import math
import re
import sys
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import sent_tokenize
from scipy.stats import zscore
from textstat import textstat as textstat_lib

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torch
except Exception as exc:
    torch = None
    print(f'Torch import failed; using CPU fallback. Error: {exc}')

try:
    from src.data.load_transcript_components import component_data_supports_qa_split
    from src.data.qa_split import flag_suspicious_qa_pairs, pair_questions_responses, split_prepared_qa
    PROJECT_QA_HELPERS_AVAILABLE = True
except Exception as exc:
    PROJECT_QA_HELPERS_AVAILABLE = False
    print(f'Project Q&A helpers unavailable; real Q&A relevance will stay optional. Error: {exc}')

DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned_yearly_announcements2.csv'
WRDS_COMPONENT_PATH = PROJECT_ROOT / 'data' / 'interim' / 'wrds_transcript_components-001.csv'
HEURISTIC_COMPONENT_PATH = PROJECT_ROOT / 'data' / 'interim' / 'transcript_components-003.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = True
RUN_REAL_QA_RELEVANCE = False
RANDOM_STATE = 42
FINBERT_BATCH_SIZE = 64
EMBEDDING_BATCH_SIZE = 32

# Fast defaults for a laptop/MPS workflow.
FULL_TRANSCRIPT_SENTIMENT_SAMPLE_SIZE = 750 if FAST_MODE else 2500
FULL_TRANSCRIPT_SENTENCE_CAP = 120 if FAST_MODE else 200
QA_RELEVANCE_MAX_TRANSCRIPTS = 600 if FAST_MODE else 2000
QA_RELEVANCE_MAX_PAIRS = 500 if FAST_MODE else 2000

PALETTE = ['#1A9E8F', '#1B2A3D', '#C0392B', '#7C3AED', '#E67E22']
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(PALETTE)

def resolve_model_device() -> str:
    if torch is None:
        return 'cpu'
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

MODEL_DEVICE = resolve_model_device()

def transformers_pipeline_device(device_name: str):
    if device_name == 'cuda':
        return 0
    if device_name == 'mps' and torch is not None:
        return torch.device('mps')
    return -1

print(f'Dataset: {DATA_PATH}')
print(f'Exists: {DATA_PATH.exists()} | Size: {DATA_PATH.stat().st_size / 1024**2:.1f} MB')
print(f'Fast mode: {FAST_MODE}')
print(f'Real Q&A relevance enabled: {RUN_REAL_QA_RELEVANCE}')
print(f'Model device: {MODEL_DEVICE}')
if torch is not None:
    print(f'CUDA available: {torch.cuda.is_available()}')
    print(f'MPS available: {getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()}')


Dataset: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/data/cleaned_yearly_announcements2.csv
Exists: True | Size: 196.0 MB
Fast mode: True
Real Q&A relevance enabled: False
Model device: mps
CUDA available: False
MPS available: True


## 1. Load the Smaller Buyback Dataset

`cleaned_yearly_announcements2.csv` already appears to be the filtered yearly buyback-announcement panel, so we load it directly instead of scanning the entire transcript universe.


In [2]:
BUYBACK_KEYWORDS = [
    r'\bbuyback\b', r'\bbuy\s*back\b', r'\brepurchas\w*\b',
    r'\bshare\s+repurchas\w*\b', r'\bstock\s+repurchas\w*\b',
    r'\brepurchase\s+program\b', r'\brepurchase\s+plan\b',
]
BUYBACK_PATTERN = re.compile('|'.join(BUYBACK_KEYWORDS), re.IGNORECASE)

REQUIRED_COLUMNS = [
    'transcriptid', 'companyid', 'companyname', 'ticker', 'call_date',
    'full_transcript_text', 'compustat_actual_revenue', 'ibes_sue_eps',
    'is_explicit_new_program', 'buyback_count', 'year',
]

header = pd.read_csv(DATA_PATH, nrows=0)
available_columns = list(header.columns)
usecols = [column for column in REQUIRED_COLUMNS if column in available_columns]
missing = sorted(set(REQUIRED_COLUMNS) - set(usecols))
print(f'Available columns ({len(available_columns)}): {available_columns}')
print(f'Using columns: {usecols}')
if missing:
    print(f'Missing requested columns: {missing}')

df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)
df['call_date'] = pd.to_datetime(df['call_date'], errors='coerce')

def extract_buyback_sentences(text: object) -> list[str]:
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return []
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [sentence.strip() for sentence in sentences if BUYBACK_PATTERN.search(sentence or '')]

start = time.time()
df['buyback_sentences'] = df['full_transcript_text'].fillna('').astype(str).map(extract_buyback_sentences)
df['buyback_sentence_count'] = df['buyback_sentences'].map(len)
df['has_buyback'] = df['buyback_sentence_count'] > 0
df['buyback_text'] = df['buyback_sentences'].map(lambda sentences: ' '.join(sentences))
bb_df = df.loc[df['has_buyback']].copy()

print('\n' + '=' * 80)
print('DATA SUMMARY')
print('=' * 80)
print(f'Rows loaded: {len(df):,}')
print(f'Date range: {df["call_date"].min()} to {df["call_date"].max()}')
print(f'Unique firms: {df["companyid"].nunique():,}')
if 'is_explicit_new_program' in df.columns:
    print('Explicit new program counts:')
    print(df['is_explicit_new_program'].value_counts(dropna=False).to_string())
if 'buyback_count' in df.columns:
    print('\nBuyback count summary:')
    print(df['buyback_count'].describe().round(2))
print(f'Buyback sentence extraction completed in {(time.time()-start):.1f}s')
print(f'Transcripts with >=1 buyback sentence: {len(bb_df):,} / {len(df):,} ({100*len(bb_df)/max(len(df),1):.1f}%)')
print(f'Median buyback sentences per call: {bb_df["buyback_sentence_count"].median():.0f}')

print('\nFIRST 2 TRANSCRIPT TEXT SNIPPETS')
for i, text in enumerate(df['full_transcript_text'].fillna('').astype(str).head(2), start=1):
    print(f'--- text sample {i} | chars={len(text):,} ---')
    print(text[:600].replace('\n', ' '))


Available columns (62): ['transcriptid', 'companyid', 'headline', 'transcriptcreationdate_utc', 'mostimportantdateutc', 'companyname', 'ticker', 'event_type', 'full_transcript_text', 'call_date', 'permno', 'actual_call_date', 'close_price_call_day', 'open_price_next_day', 'close_to_open_return', 'ret_t-15', 'ret_t-14', 'ret_t-13', 'ret_t-12', 'ret_t-11', 'ret_t-10', 'ret_t-9', 'ret_t-8', 'ret_t-7', 'ret_t-6', 'ret_t-5', 'ret_t-4', 'ret_t-3', 'ret_t-2', 'ret_t-1', 'ret_t0', 'ret_t1', 'ret_t2', 'ret_t3', 'ret_t4', 'ret_t5', 'ret_t6', 'ret_t7', 'ret_t8', 'ret_t9', 'ret_t10', 'ret_t11', 'ret_t12', 'ret_t13', 'ret_t14', 'ret_t15', 'transcript_length', 'word_count', 'gvkey', 'fiscal_period_end', 'report_date', 'fiscal_year', 'fiscal_quarter', 'compustat_actual_revenue', 'ibes_ticker', 'ibes_anndats', 'ibes_mean_est_eps', 'ibes_actual_eps', 'ibes_raw_surp_eps', 'ibes_sue_eps', 'is_explicit_new_program', 'year']
Using columns: ['transcriptid', 'companyid', 'companyname', 'ticker', 'call_date',

In [3]:
fig, ax = plt.subplots(figsize=(10, 5))
bb_df['buyback_sentence_count'].clip(upper=30).hist(
    bins=30, ax=ax, color='#1A9E8F', edgecolor='white', alpha=0.85
)
median_mentions = bb_df['buyback_sentence_count'].median()
ax.axvline(median_mentions, color='#C0392B', linestyle='--', linewidth=2, label=f'Median: {median_mentions:.0f}')
ax.set_xlabel('Buyback Mentions per Call', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Buyback Mentions per Earnings Call', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
out_path = OUTPUT_DIR / 'buyback_mention_frequency.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/buyback_mention_frequency.png


## 2. Runtime Estimate Before Scoring

This cell gives a rough sense of how long the default fast run should take on a laptop. It is only an estimate, but it helps set expectations before loading models.


In [4]:
sentence_count_sample = []
for text in bb_df['full_transcript_text'].fillna('').astype(str).head(min(300, len(bb_df))):
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    sentence_count_sample.append(len([sentence for sentence in sentences if str(sentence).strip()]))

sentence_count_sample = pd.Series(sentence_count_sample)
avg_buyback_sentences = bb_df['buyback_sentence_count'].mean()
avg_full_sentences_capped = sentence_count_sample.clip(upper=FULL_TRANSCRIPT_SENTENCE_CAP).mean()
est_buyback_sentences_total = int(round(len(bb_df) * avg_buyback_sentences))
est_full_sentences_total = int(round(min(FULL_TRANSCRIPT_SENTIMENT_SAMPLE_SIZE, len(bb_df)) * avg_full_sentences_capped))
est_total_sentences = est_buyback_sentences_total + est_full_sentences_total

# Conservative-through-optimistic range for local MPS/CPU runs.
slow_rate = 2500
fast_rate = 6000
est_minutes_slow = est_total_sentences / slow_rate
est_minutes_fast = est_total_sentences / fast_rate

print('Sentence volume estimate')
print(f'- Avg buyback sentences/call: {avg_buyback_sentences:.1f}')
print(f'- Avg capped full-transcript sentences/call: {avg_full_sentences_capped:.1f}')
print(f'- Estimated buyback-specific sentences to score: {est_buyback_sentences_total:,}')
print(f'- Estimated full-transcript sentences to score: {est_full_sentences_total:,}')
print(f'- Estimated total sentences to score: {est_total_sentences:,}')
print(f'- Rough total runtime for default fast mode: {est_minutes_fast:.0f}-{est_minutes_slow:.0f} minutes on this Mac')
if RUN_REAL_QA_RELEVANCE:
    print('- Add roughly 10-25 extra minutes if real component-based Q&A relevance is enabled.')
else:
    print('- Real component-based Q&A relevance is off by default, which saves significant time.')


Sentence volume estimate
- Avg buyback sentences/call: 5.7
- Avg capped full-transcript sentences/call: 120.0
- Estimated buyback-specific sentences to score: 26,041
- Estimated full-transcript sentences to score: 90,000
- Estimated total sentences to score: 116,041
- Rough total runtime for default fast mode: 19-46 minutes on this Mac
- Real component-based Q&A relevance is off by default, which saves significant time.


## 3. FinBERT Sentiment

The fast default keeps the buyback-specific sentiment pass across all rows, but limits the full-transcript comparison to a smaller sample and a lower sentence cap.


In [5]:
from transformers import pipeline

FINBERT_MODEL_NAME = 'ProsusAI/finbert'
FINBERT_DEVICE = MODEL_DEVICE

def load_finbert_pipeline(device_name: str):
    return pipeline(
        'sentiment-analysis',
        model=FINBERT_MODEL_NAME,
        tokenizer=FINBERT_MODEL_NAME,
        device=transformers_pipeline_device(device_name),
        batch_size=FINBERT_BATCH_SIZE,
        truncation=True,
        max_length=512,
    )

try:
    finbert = load_finbert_pipeline(FINBERT_DEVICE)
except Exception as exc:
    print(f'FinBERT load failed on {FINBERT_DEVICE}: {exc}')
    print('Retrying on CPU...')
    FINBERT_DEVICE = 'cpu'
    finbert = load_finbert_pipeline(FINBERT_DEVICE)

print(f'FinBERT loaded: {FINBERT_MODEL_NAME} on {FINBERT_DEVICE}')

def signed_finbert_scores(texts: list[str]) -> list[float]:
    if not texts:
        return []
    predictions = finbert(texts, batch_size=FINBERT_BATCH_SIZE, truncation=True, max_length=512)
    scores = []
    for prediction in predictions:
        label = str(prediction.get('label', '')).lower()
        confidence = float(prediction.get('score', 0.0))
        if label == 'positive':
            scores.append(confidence)
        elif label == 'negative':
            scores.append(-confidence)
        else:
            scores.append(0.0)
    return scores

def aggregate_scores(scores: list[float]) -> dict[str, float]:
    if not scores:
        return {'mean': np.nan, 'p10': np.nan, 'min': np.nan, 'std': np.nan, 'n_sentences': 0}
    arr = np.asarray(scores, dtype=float)
    return {
        'mean': float(np.mean(arr)),
        'p10': float(np.percentile(arr, 10)),
        'min': float(np.min(arr)),
        'std': float(np.std(arr)),
        'n_sentences': int(len(arr)),
    }


Device set to use mps


FinBERT loaded: ProsusAI/finbert on mps


In [6]:
print('Scoring buyback-specific sentences for all rows...')
start = time.time()
flat_sentences = []
flat_row_ids = []
for row_id, sentences in enumerate(bb_df['buyback_sentences']):
    for sentence in sentences:
        flat_row_ids.append(row_id)
        flat_sentences.append(sentence)

print(f'Buyback-specific sentences to score: {len(flat_sentences):,}')
flat_scores = signed_finbert_scores(flat_sentences)
grouped_scores = [[] for _ in range(len(bb_df))]
for row_id, score in zip(flat_row_ids, flat_scores):
    grouped_scores[row_id].append(score)

bb_df['buyback_sentiment_scores'] = grouped_scores
bb_df['buyback_sentiment_mean'] = [aggregate_scores(scores)['mean'] for scores in grouped_scores]
bb_df['buyback_sentiment_p10'] = [aggregate_scores(scores)['p10'] for scores in grouped_scores]
bb_df['buyback_sentiment_min'] = [aggregate_scores(scores)['min'] for scores in grouped_scores]
bb_df['buyback_sentiment_std'] = [aggregate_scores(scores)['std'] for scores in grouped_scores]

print(f'Buyback sentiment scoring completed in {(time.time() - start)/60:.1f} minutes')
print(bb_df[['buyback_sentence_count', 'buyback_sentiment_mean', 'buyback_sentiment_p10']].describe().round(4))


Scoring buyback-specific sentences for all rows...
Buyback-specific sentences to score: 26,041
Buyback sentiment scoring completed in 10.9 minutes
       buyback_sentence_count  buyback_sentiment_mean  buyback_sentiment_p10
count               4601.0000               4601.0000              4601.0000
mean                   5.6599                  0.3173                 0.0246
std                    3.6507                  0.2508                 0.3022
min                    1.0000                 -0.9630                -0.9761
25%                    3.0000                  0.1515                 0.0000
50%                    5.0000                  0.3020                 0.0000
75%                    7.0000                  0.4650                 0.0000
max                   34.0000                  0.9532                 0.9532


In [7]:
sample_size = min(FULL_TRANSCRIPT_SENTIMENT_SAMPLE_SIZE, len(bb_df))
sample_df = bb_df.sample(n=sample_size, random_state=RANDOM_STATE).copy()
print(f'Scoring full-transcript sentiment for {sample_size:,} sampled rows...')
print(f'Sentence cap per transcript: {FULL_TRANSCRIPT_SENTENCE_CAP}')

full_scores_by_index = defaultdict(list)
batch_texts = []
batch_indices = []
start = time.time()

def flush_full_sentiment_batch():
    if not batch_texts:
        return
    scores = signed_finbert_scores(batch_texts)
    for idx, score in zip(batch_indices, scores):
        full_scores_by_index[idx].append(score)
    batch_texts.clear()
    batch_indices.clear()

for transcript_number, (idx, row) in enumerate(sample_df.iterrows(), start=1):
    text = row.get('full_transcript_text', '')
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        continue
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    for sentence in sentences[:FULL_TRANSCRIPT_SENTENCE_CAP]:
        sentence = str(sentence).strip()
        if not sentence:
            continue
        batch_indices.append(idx)
        batch_texts.append(sentence)
        if len(batch_texts) >= FINBERT_BATCH_SIZE:
            flush_full_sentiment_batch()
    if transcript_number == 1 or transcript_number % 250 == 0:
        print(f'Processed {transcript_number:,}/{sample_size:,} transcripts | elapsed {(time.time()-start)/60:.1f} min')

flush_full_sentiment_batch()
sample_df['full_transcript_sentiment'] = sample_df.index.to_series().map(
    {idx: float(np.mean(scores)) for idx, scores in full_scores_by_index.items() if scores}
)
sample_df['sentiment_gap'] = sample_df['buyback_sentiment_mean'] - sample_df['full_transcript_sentiment']

bb_df['full_transcript_sentiment'] = np.nan
bb_df['sentiment_gap'] = np.nan
bb_df.loc[sample_df.index, 'full_transcript_sentiment'] = sample_df['full_transcript_sentiment']
bb_df.loc[sample_df.index, 'sentiment_gap'] = sample_df['sentiment_gap']

print(f'Full-transcript sample sentiment completed in {(time.time() - start)/60:.1f} minutes')
print(sample_df[['buyback_sentiment_mean', 'full_transcript_sentiment', 'sentiment_gap']].describe().round(4))


Scoring full-transcript sentiment for 750 sampled rows...
Sentence cap per transcript: 120
Processed 1/750 transcripts | elapsed 0.0 min
Processed 250/750 transcripts | elapsed 8.3 min
Processed 500/750 transcripts | elapsed 17.0 min
Processed 750/750 transcripts | elapsed 25.5 min
Full-transcript sample sentiment completed in 25.6 minutes
       buyback_sentiment_mean  full_transcript_sentiment  sentiment_gap
count                750.0000                   750.0000       750.0000
mean                   0.3215                     0.2809         0.0406
std                    0.2486                     0.1318         0.2625
min                   -0.9630                    -0.1321        -1.0168
25%                    0.1559                     0.1939        -0.1365
50%                    0.3135                     0.2812         0.0247
75%                    0.4617                     0.3698         0.1896
max                    0.9469                     0.6490         0.9753


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sample_df['full_transcript_sentiment'].dropna(), bins=50, alpha=0.6, label='Full Transcript', color='#1B2A3D', density=True)
axes[0].hist(sample_df['buyback_sentiment_mean'].dropna(), bins=50, alpha=0.6, label='Buyback-Specific', color='#1A9E8F', density=True)
axes[0].set_xlabel('FinBERT Sentiment Score', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Sentiment: Full Transcript vs. Buyback-Specific', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].axvline(0, color='gray', linestyle=':', alpha=0.5)

gap = sample_df['sentiment_gap'].dropna()
axes[1].hist(gap, bins=50, color='#7C3AED', alpha=0.75, edgecolor='white')
if not gap.empty:
    axes[1].axvline(gap.median(), color='#C0392B', linestyle='--', linewidth=2, label=f'Median: {gap.median():.3f}')
axes[1].axvline(0, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Buyback Sentiment Gap', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Buyback Sentiment Gap Distribution', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
out_path = OUTPUT_DIR / 'sentiment_distributions.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/sentiment_distributions.png


In [9]:
sample_df['year_plot'] = pd.to_datetime(sample_df['call_date'], errors='coerce').dt.year
yearly_sent = sample_df.groupby('year_plot').agg(
    mean_full=('full_transcript_sentiment', 'mean'),
    mean_buyback=('buyback_sentiment_mean', 'mean'),
    count=('buyback_sentiment_mean', 'count'),
).reset_index().dropna(subset=['year_plot'])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(yearly_sent['year_plot'], yearly_sent['mean_full'], marker='o', linewidth=2, label='Full Transcript', color='#1B2A3D')
ax1.plot(yearly_sent['year_plot'], yearly_sent['mean_buyback'], marker='s', linewidth=2, label='Buyback-Specific', color='#1A9E8F')
ax1.set_xlabel('Year', fontsize=11)
ax1.set_ylabel('Mean Sentiment', fontsize=11)
ax1.set_title('Mean Sentiment by Year', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax2 = ax1.twinx()
ax2.bar(yearly_sent['year_plot'], yearly_sent['count'], alpha=0.15, color='#1A9E8F')
ax2.set_ylabel('Number of Events', fontsize=10, color='gray')
plt.tight_layout()
out_path = OUTPUT_DIR / 'sentiment_by_year.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/sentiment_by_year.png


In [10]:
examples = []
for _, row in bb_df.head(250).iterrows():
    sentences = row.get('buyback_sentences', [])
    scores = row.get('buyback_sentiment_scores', [])
    for sentence, score in list(zip(sentences, scores))[:2]:
        examples.append({'sentence': sentence, 'score': score})
    if len(examples) >= 40:
        break

examples_df = pd.DataFrame(examples).dropna(subset=['score']).sort_values('score').reset_index(drop=True)
print('\n' + '=' * 80)
print('EXAMPLE SENTENCES WITH FINBERT SCORES')
print('=' * 80)
if not examples_df.empty:
    neg = examples_df.iloc[0]
    examples_df['abs_score'] = examples_df['score'].abs()
    neut = examples_df.sort_values('abs_score').iloc[0]
    pos = examples_df.sort_values('score').iloc[-1]
    print(f'\nNEGATIVE (score={neg["score"]:.4f}):')
    print(f'  "{neg["sentence"][:240]}"')
    print(f'\nNEUTRAL (score={neut["score"]:.4f}):')
    print(f'  "{neut["sentence"][:240]}"')
    print(f'\nPOSITIVE (score={pos["score"]:.4f}):')
    print(f'  "{pos["sentence"][:240]}"')

out_path = OUTPUT_DIR / 'sentiment_examples.csv'
examples_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')



EXAMPLE SENTENCES WITH FINBERT SCORES

NEGATIVE (score=-0.9741):
  "At quarter end, cash totaled $357 million, sequentially lower by $31 million due in part to our investments in working capital, capital expenditures and our expanded share repurchase program."

NEUTRAL (score=0.0000):
  "Over the last 12 months, we've repurchased more than 5% of the shares outstanding a year ago and paid out more than $2.1 billion in dividend, as nearly 75% of our cash flow was returned to shareholders."

POSITIVE (score=0.9376):
  "We plan to accelerate our share repurchase activity in the coming year beyond these levels to increase our return to shareholders."

Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/sentiment_examples.csv


## 4. Clarity Features

The default clarity workflow uses three fast components: specificity, hedge density, and modified FOG. Real Q&A relevance is optional and off by default for speed.


In [11]:
FINANCIAL_EXCLUSIONS = {
    'company', 'companies', 'management', 'operating', 'financial', 'securities',
    'investment', 'investments', 'performance', 'opportunity', 'opportunities',
    'infrastructure', 'capitalize', 'approximately', 'international', 'administration',
    'environment', 'executive', 'executives', 'generation', 'acquisition', 'acquisitions',
    'technology', 'technologies', 'significant', 'significantly', 'competitive',
    'operations', 'operational', 'continuing', 'outstanding', 'delivered', 'delivering',
    'percentage', 'revenue', 'revenues', 'dividend', 'dividends', 'leverage',
    'enterprise', 'inventory', 'marketing', 'customer', 'customers', 'improvement',
    'improvements', 'industry', 'industries', 'commercial', 'manufacture', 'manufacturing',
    'contribute', 'contribution', 'regulatory', 'implementing', 'implementation',
    'development', 'developing', 'commitment', 'committed', 'distributing', 'distribution',
}

HEDGE_WORDS = {
    'may', 'might', 'could', 'possibly', 'perhaps', 'potentially', 'uncertain',
    'uncertainty', 'unclear', 'unpredictable', 'approximate', 'approximately',
    'volatile', 'volatility', 'risk', 'risks', 'risky', 'conceivably',
    'presumably', 'indefinite', 'contingent', 'speculative', 'tentative',
    'believe', 'think', 'feel', 'hope', 'expect', 'anticipate', 'intend',
    'appear', 'suggest', 'seem', 'likely', 'possible', 'probable',
    'premature', 'eventually', 'someday',
}

def compute_specificity(text: object) -> int:
    if pd.isna(text) or not isinstance(text, str):
        return 0
    score = 0
    if re.search(r'\$[\d,.]+\s*(billion|million|B|M|bn|mn)?', text, re.IGNORECASE):
        score += 1
    if re.search(r'[\d,.]+\s*(shares|million\s+shares|billion\s+shares)', text, re.IGNORECASE):
        score += 1
    if re.search(r'(Q[1-4]\s*\d{4}|through\s+\d{4}|over\s+the\s+next|by\s+(year|end|mid)|fiscal\s+year|\d+\s*(year|month|quarter))', text, re.IGNORECASE):
        score += 1
    if re.search(r'(free\s+cash\s+flow|cash\s+on\s+hand|operating\s+cash|debt|credit\s+facility|balance\s+sheet|cash\s+generation|excess\s+cash)', text, re.IGNORECASE):
        score += 1
    return score

def compute_hedge_density(text: object) -> float:
    if pd.isna(text) or not isinstance(text, str):
        return 0.0
    words = re.findall(r'\b\w+\b', text.lower())
    if not words:
        return 0.0
    hedge_count = sum(1 for word in words if word in HEDGE_WORDS)
    return float(hedge_count / len(words))

def compute_modified_fog(text: object) -> float:
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 20:
        return np.nan
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    total_words = len(words)
    if total_words == 0:
        return np.nan
    complex_words_standard = sum(1 for word in words if textstat_lib.syllable_count(word) >= 3)
    complex_words_excluded = sum(
        1 for word in words
        if textstat_lib.syllable_count(word) >= 3 and word in FINANCIAL_EXCLUSIONS
    )
    adjusted_complex = complex_words_standard - complex_words_excluded
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    n_sentences = max(len([sentence for sentence in sentences if str(sentence).strip()]), 1)
    words_per_sentence = total_words / n_sentences
    return float(0.4 * (words_per_sentence + 100 * max(adjusted_complex, 0) / total_words))

def compute_standard_fog(text: object) -> float:
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 20:
        return np.nan
    return float(textstat_lib.gunning_fog(text))


In [12]:
start = time.time()
bb_df['specificity'] = bb_df['buyback_text'].map(compute_specificity)
bb_df['hedge_density'] = bb_df['buyback_text'].map(compute_hedge_density)
bb_df['modified_fog'] = bb_df['buyback_text'].map(compute_modified_fog)
bb_df['standard_fog'] = bb_df['buyback_text'].map(compute_standard_fog)
bb_df['qa_relevance'] = np.nan

print(f'Clarity feature extraction completed in {time.time()-start:.1f}s')
print('Specificity distribution:')
print(bb_df['specificity'].value_counts().sort_index())
print(f'Hedge density mean: {bb_df["hedge_density"].mean():.4f}')
print(f'Standard FOG mean: {bb_df["standard_fog"].mean():.2f}')
print(f'Modified FOG mean: {bb_df["modified_fog"].mean():.2f}')
print(f'Reduction: {bb_df["standard_fog"].mean() - bb_df["modified_fog"].mean():.2f} grade levels')


Clarity feature extraction completed in 4.7s
Specificity distribution:
specificity
0     207
1    1356
2    1813
3     997
4     228
Name: count, dtype: int64
Hedge density mean: 0.0115
Standard FOG mean: 15.29
Modified FOG mean: 16.26
Reduction: -0.97 grade levels


## 5. Optional Q&A Relevance

The notebook always writes the demo text file for presentation. The real component-based relevance pass is optional because loading the component files is one of the slowest parts of the workflow.


In [13]:
BGE_PRIMARY_MODEL = 'BAAI/bge-large-en-v1.5'
BGE_FALLBACK_MODEL = 'BAAI/bge-base-en-v1.5'

def load_embedding_model():
    from sentence_transformers import SentenceTransformer
    for model_name in [BGE_PRIMARY_MODEL, BGE_FALLBACK_MODEL]:
        try:
            print(f'Loading embedding model {model_name} on {MODEL_DEVICE}...')
            model = SentenceTransformer(model_name, device=MODEL_DEVICE)
            print(f'Loaded embedding model: {model_name}')
            return model, model_name
        except Exception as exc:
            print(f'Embedding model load failed for {model_name}: {exc}')
    return None, None

def cosine_similarity_pairs(model, questions: list[str], answers: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> list[float]:
    if model is None or not questions:
        return []
    scores = []
    for start_idx in range(0, len(questions), batch_size):
        q_batch = questions[start_idx:start_idx + batch_size]
        a_batch = answers[start_idx:start_idx + batch_size]
        q_emb = model.encode(q_batch, normalize_embeddings=True, convert_to_numpy=True)
        a_emb = model.encode(a_batch, normalize_embeddings=True, convert_to_numpy=True)
        scores.extend(np.sum(q_emb * a_emb, axis=1).astype(float).tolist())
    return scores

embed_model = None
embed_model_name = None
if RUN_REAL_QA_RELEVANCE:
    embed_model, embed_model_name = load_embedding_model()
else:
    print('Real Q&A relevance is disabled in fast mode. Set RUN_REAL_QA_RELEVANCE=True to enable it.')


Real Q&A relevance is disabled in fast mode. Set RUN_REAL_QA_RELEVANCE=True to enable it.


In [14]:
demo_qa_pairs = [
    {
        'question': 'Can you give us more detail on the buyback authorization and how you plan to fund it?',
        'answer_direct': 'Sure. We authorized a $2 billion repurchase program through the end of fiscal 2025, funded entirely from free cash flow. We expect to execute approximately $500 million per quarter.',
        'answer_evasive': "Great question. We're really excited about returning capital to shareholders. Our board has authorized a program and we'll be opportunistic in our approach going forward.",
    },
    {
        'question': 'What is driving the increase in your share repurchase activity this quarter?',
        'answer_direct': 'Two factors. First, our stock is trading at 11x forward earnings, which is a 30% discount to our 5-year average. Second, we generated $800 million in free cash flow this quarter, well above our capital needs.',
        'answer_evasive': "We believe our shares represent good value and we're committed to a balanced capital allocation strategy that includes returning value to shareholders.",
    },
]

demo_lines = []
if RUN_REAL_QA_RELEVANCE and embed_model is not None:
    demo_lines.append(f'Embedding model used: {embed_model_name}\n\n')
    for i, pair in enumerate(demo_qa_pairs, start=1):
        sim_direct, sim_evasive = cosine_similarity_pairs(
            embed_model,
            [pair['question'], pair['question']],
            [pair['answer_direct'], pair['answer_evasive']],
        )
        demo_lines.extend([
            f'Pair {i}:\n',
            f'  Q: {pair["question"]}\n',
            f'  Direct similarity: {sim_direct:.4f}\n',
            f'  Evasive similarity: {sim_evasive:.4f}\n',
            f'  Gap: {sim_direct - sim_evasive:+.4f}\n\n',
        ])
else:
    demo_lines.append('Real embedding model run was skipped in fast mode.\n')
    demo_lines.append('Presentation logic: a direct answer should have higher cosine similarity to the question than an evasive answer.\n\n')
    for i, pair in enumerate(demo_qa_pairs, start=1):
        demo_lines.extend([
            f'Pair {i}:\n',
            f'  Q: {pair["question"]}\n',
            f'  Direct answer: {pair["answer_direct"]}\n',
            f'  Evasive answer: {pair["answer_evasive"]}\n\n',
        ])

out_path = OUTPUT_DIR / 'qa_relevance_demo.txt'
out_path.write_text(''.join(demo_lines), encoding='utf-8')
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/qa_relevance_demo.txt


## 6. Clarity Composite & Distributions


In [15]:
def safe_zscore(series: pd.Series, fill_value: float | None = None) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    if fill_value is None:
        fill_value = numeric.median() if numeric.notna().any() else 0.0
    filled = numeric.fillna(fill_value)
    if filled.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(filled)), index=series.index)
    return pd.Series(zscore(filled), index=series.index).replace([np.inf, -np.inf], 0).fillna(0)

bb_df['z_specificity'] = safe_zscore(bb_df['specificity'], fill_value=0)
bb_df['z_hedge'] = safe_zscore(bb_df['hedge_density'], fill_value=0)
bb_df['z_fog'] = safe_zscore(bb_df['modified_fog'])
bb_df['clarity_composite'] = (bb_df['z_specificity'] - bb_df['z_hedge'] - bb_df['z_fog']) / 3

print(f'Clarity composite mean: {bb_df["clarity_composite"].mean():.4f}')
print(f'Clarity composite std: {bb_df["clarity_composite"].std():.4f}')


Clarity composite mean: -0.0000
Clarity composite std: 0.5425


In [16]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(bb_df['specificity'].dropna(), bins=5, color='#1A9E8F', edgecolor='white', alpha=0.85, rwidth=0.85)
axes[0, 0].set_title('Specificity Score (0-4)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Score')

axes[0, 1].hist(bb_df['hedge_density'].dropna(), bins=40, color='#C0392B', edgecolor='white', alpha=0.85)
axes[0, 1].set_title('Hedge Density (inverted for composite)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Hedge Words / Total Words')

axes[1, 0].hist(bb_df['modified_fog'].dropna().clip(0, 30), bins=40, color='#E67E22', edgecolor='white', alpha=0.85, label='Modified FOG')
axes[1, 0].hist(bb_df['standard_fog'].dropna().clip(0, 30), bins=40, color='#1B2A3D', edgecolor='white', alpha=0.3, label='Standard FOG')
axes[1, 0].set_title('Modified FOG Index', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Grade Level')
axes[1, 0].legend(fontsize=9)

axes[1, 1].hist(bb_df['clarity_composite'].dropna(), bins=40, color='#7C3AED', edgecolor='white', alpha=0.85)
axes[1, 1].set_title('Clarity Composite (z-scored)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Composite Score')
p33 = bb_df['clarity_composite'].quantile(0.333)
p67 = bb_df['clarity_composite'].quantile(0.667)
axes[1, 1].axvline(p33, color='#C0392B', linestyle='--', label=f'P33: {p33:.2f}')
axes[1, 1].axvline(p67, color='#1A9E8F', linestyle='--', label=f'P67: {p67:.2f}')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Clarity Component Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = OUTPUT_DIR / 'clarity_distributions.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/clarity_distributions.png


In [17]:
corr_matrix = bb_df[['specificity', 'hedge_density', 'modified_fog']].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=1,
    ax=ax,
    xticklabels=['Specificity', 'Hedge Density', 'Modified FOG'],
    yticklabels=['Specificity', 'Hedge Density', 'Modified FOG'],
)
ax.set_title('Clarity Component Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = OUTPUT_DIR / 'clarity_correlation_matrix.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/clarity_correlation_matrix.png


In [18]:
fig, ax = plt.subplots(figsize=(8, 6))
valid_fog = bb_df[['standard_fog', 'modified_fog']].dropna()
ax.scatter(valid_fog['standard_fog'], valid_fog['modified_fog'], alpha=0.15, s=8, color='#1A9E8F')
ax.plot([0, 30], [0, 30], 'r--', alpha=0.5, label='y = x (no adjustment)')
ax.set_xlabel('Standard Gunning FOG', fontsize=11)
ax.set_ylabel('Modified FOG', fontsize=11)
ax.set_title('Effect of Financial Vocabulary Exclusion on FOG Index', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 30)
ax.set_ylim(0, 30)
plt.tight_layout()
out_path = OUTPUT_DIR / 'fog_standard_vs_modified.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/fog_standard_vs_modified.png


## 7. 3x3 Binning and Summary Outputs

The sentiment and clarity buckets below are now built without lookahead bias:
- estimate the bucket cutoffs using only 2010-2012 data
- apply those fixed cutoffs to 2013-2014 data
- build the 3x3 matrix on the 2013-2014 out-of-sample subset


In [19]:
TRAIN_YEARS = (2010, 2012)
APPLY_YEARS = (2013, 2014)

bb_df['call_date'] = pd.to_datetime(bb_df['call_date'], errors='coerce')
bb_df['bucket_year'] = bb_df['call_date'].dt.year

train_mask = bb_df['bucket_year'].between(TRAIN_YEARS[0], TRAIN_YEARS[1], inclusive='both')
apply_mask = bb_df['bucket_year'].between(APPLY_YEARS[0], APPLY_YEARS[1], inclusive='both')

train_df = bb_df.loc[train_mask].copy()
apply_df = bb_df.loc[apply_mask].copy()

print(f'Training window: {TRAIN_YEARS[0]}-{TRAIN_YEARS[1]} | rows: {len(train_df):,}')
print(f'Application window: {APPLY_YEARS[0]}-{APPLY_YEARS[1]} | rows: {len(apply_df):,}')


def fit_tercile_cutoffs(series: pd.Series) -> tuple[float, float]:
    values = pd.to_numeric(series, errors='coerce').dropna().astype(float)
    if len(values) < 3:
        raise ValueError('Need at least 3 non-null observations to fit tercile cutoffs.')
    q1 = float(values.quantile(1/3))
    q2 = float(values.quantile(2/3))
    if q2 < q1:
        q1, q2 = q2, q1
    return q1, q2


def apply_fixed_buckets(series: pd.Series, lower_cut: float, upper_cut: float, labels: list[str]) -> pd.Categorical:
    numeric = pd.to_numeric(series, errors='coerce')
    result = pd.Series(pd.NA, index=series.index, dtype='object')
    result.loc[numeric.notna() & (numeric <= lower_cut)] = labels[0]
    result.loc[numeric.notna() & (numeric > lower_cut) & (numeric <= upper_cut)] = labels[1]
    result.loc[numeric.notna() & (numeric > upper_cut)] = labels[2]
    return pd.Categorical(result, categories=labels, ordered=True)

sentiment_cut_1, sentiment_cut_2 = fit_tercile_cutoffs(train_df['buyback_sentiment_mean'])
clarity_cut_1, clarity_cut_2 = fit_tercile_cutoffs(train_df['clarity_composite'])

print('\nFixed training-period cutoffs')
print(f'Sentiment cutoffs: <= {sentiment_cut_1:.4f} = Negative | <= {sentiment_cut_2:.4f} = Neutral | > {sentiment_cut_2:.4f} = Positive')
print(f'Clarity cutoffs:   <= {clarity_cut_1:.4f} = Low      | <= {clarity_cut_2:.4f} = Medium  | > {clarity_cut_2:.4f} = High')

bb_df['sentiment_tercile'] = apply_fixed_buckets(
    bb_df['buyback_sentiment_mean'],
    sentiment_cut_1,
    sentiment_cut_2,
    ['Negative', 'Neutral', 'Positive'],
)
bb_df['clarity_tercile'] = apply_fixed_buckets(
    bb_df['clarity_composite'],
    clarity_cut_1,
    clarity_cut_2,
    ['Low', 'Medium', 'High'],
)

apply_scored = bb_df.loc[
    apply_mask & bb_df['sentiment_tercile'].notna() & bb_df['clarity_tercile'].notna()
].copy()

ct = pd.crosstab(apply_scored['sentiment_tercile'], apply_scored['clarity_tercile'])
ct = ct.reindex(index=['Negative', 'Neutral', 'Positive'], columns=['Low', 'Medium', 'High'], fill_value=0)

print('\n3x3 Matrix - Cell Counts (cutoffs fit on 2010-2012, applied to 2013-2014):')
print(ct)
print(f'\nMinimum cell count: {ct.min().min()}')
print(f'All cells >= 30: {(ct >= 30).all().all()}')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', linewidths=1, ax=ax, cbar_kws={'label': 'N observations'})
ax.set_title('3x3 Matrix - 2013-2014 OOS Cell Counts', fontsize=13, fontweight='bold')
ax.set_xlabel('Clarity Bucket (fit on 2010-2012)', fontsize=11)
ax.set_ylabel('Sentiment Bucket (fit on 2010-2012)', fontsize=11)
plt.tight_layout()
out_path = OUTPUT_DIR / 'matrix_cell_counts.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')



3x3 Matrix - Cell Counts:
clarity_tercile    Low  Medium  High
sentiment_tercile                   
Negative           467     505   562
Neutral            487     476   570
Positive           580     552   402

Minimum cell count: 402
All cells >= 30: True
Saved: /Users/shahmirjaved/Documents/Projects/Analytical Finance and Machine Learning/Analytical-Finance-and-Machine-Learning/outputs/eda/matrix_cell_counts.png


In [20]:
summary_cols = [
    'buyback_sentiment_mean', 'buyback_sentiment_p10', 'sentiment_gap',
    'specificity', 'hedge_density', 'modified_fog', 'standard_fog',
    'clarity_composite', 'buyback_sentence_count', 'buyback_count',
]
summary_cols = [column for column in summary_cols if column in bb_df.columns]
summary = bb_df[summary_cols].describe().round(4)
summary_path = OUTPUT_DIR / 'summary_statistics.csv'
summary.to_csv(summary_path)
print(summary)
print(f'Saved: {summary_path}')

enriched_path = INTERIM_DIR / 'buyback_transcripts_with_features.csv'
bb_df.to_csv(enriched_path, index=False)
print(f'Saved enriched dataset: {enriched_path} ({len(bb_df):,} rows)')


       buyback_sentiment_mean  buyback_sentiment_p10  sentiment_gap  \
count               4601.0000              4601.0000       750.0000   
mean                   0.3173                 0.0246         0.0406   
std                    0.2508                 0.3022         0.2625   
min                   -0.9630                -0.9761        -1.0168   
25%                    0.1515                 0.0000        -0.1365   
50%                    0.3020                 0.0000         0.0247   
75%                    0.4650                 0.0000         0.1896   
max                    0.9532                 0.9532         0.9753   

       specificity  hedge_density  modified_fog  standard_fog  \
count    4601.0000      4601.0000     4601.0000     4601.0000   
mean        1.9311         0.0115       16.2622       15.2915   
std         0.9408         0.0117        3.3670        3.5711   
min         0.0000         0.0000        3.6000        3.6000   
25%         1.0000         0.0000  

In [21]:
expected_outputs = [
    OUTPUT_DIR / 'buyback_mention_frequency.png',
    OUTPUT_DIR / 'sentiment_distributions.png',
    OUTPUT_DIR / 'sentiment_by_year.png',
    OUTPUT_DIR / 'sentiment_examples.csv',
    OUTPUT_DIR / 'qa_relevance_demo.txt',
    OUTPUT_DIR / 'clarity_distributions.png',
    OUTPUT_DIR / 'clarity_correlation_matrix.png',
    OUTPUT_DIR / 'fog_standard_vs_modified.png',
    OUTPUT_DIR / 'matrix_cell_counts.png',
    OUTPUT_DIR / 'summary_statistics.csv',
    INTERIM_DIR / 'buyback_transcripts_with_features.csv',
]

print('\n' + '=' * 80)
print('EXPECTED OUTPUT CHECK')
print('=' * 80)
missing = []
for path in expected_outputs:
    exists = path.exists() and path.stat().st_size > 0
    status = 'OK' if exists else 'MISSING'
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f'  {status:7s} {path.relative_to(PROJECT_ROOT)} ({size_kb:.1f} KB)')
    if not exists:
        missing.append(path)
if missing:
    print('\nMissing or empty outputs remain:')
    for path in missing:
        print(f'  - {path.relative_to(PROJECT_ROOT)}')
else:
    print('\nAll expected outputs exist and are non-empty.')



EXPECTED OUTPUT CHECK
  OK      outputs/eda/buyback_mention_frequency.png (57.5 KB)
  OK      outputs/eda/sentiment_distributions.png (102.3 KB)
  OK      outputs/eda/sentiment_by_year.png (120.5 KB)
  OK      outputs/eda/sentiment_examples.csv (7.5 KB)
  OK      outputs/eda/qa_relevance_demo.txt (1.1 KB)
  OK      outputs/eda/clarity_distributions.png (170.3 KB)
  OK      outputs/eda/clarity_correlation_matrix.png (63.9 KB)
  OK      outputs/eda/fog_standard_vs_modified.png (319.5 KB)
  OK      outputs/eda/matrix_cell_counts.png (73.3 KB)
  OK      outputs/eda/summary_statistics.csv (0.6 KB)
  OK      data/interim/buyback_transcripts_with_features.csv (207207.7 KB)

All expected outputs exist and are non-empty.
